# RSL-RL Legged vs Wheeled Performance

Analyze the **16-condition × 3-seed** morphology campaign (all at `--max_iterations=20000`, `--num_envs=1024`):

| Pair | Legged | Wheeled | Terrains |
|------|--------|---------|----------|
| Go2 | `unitree_go2_{flat,rough}` | `unitree_go2w_{flat,rough}` | flat, rough |
| B2 | `unitree_b2_{flat,rough}` | `unitree_b2w_{flat,rough}` | flat, rough |
| ZSL1 | `zsibot_zsl1_{flat,rough}` | `zsibot_zsl1w_{flat,rough}` | flat, rough |
| Lite3 / M20 | `deeprobotics_lite3_{flat,rough}` | `deeprobotics_m20_{flat,rough}` | flat, rough |

**Campaign:** 4 pairs × 2 morphologies × 2 terrains × 3 seeds (`42` / `0` / `1`) = **48** runs.

**Important:** only seed-tagged runs (`*_seed{N}`) for these 16 experiments are loaded. Table I (1500-iter) logs are excluded by experiment filter. Older / untagged logs are ignored.

**Primary metrics:** `mean_reward`, `track_lin_vel`, `track_ang_vel`, `episode_length`.

**Stage learning:** reward snapshots and stage rates at **25 / 50 / 75 / 100%** of the 20k-iteration budget (steps 5k / 10k / 15k / 20k). Relative `steps_to_75pct` (self-normalized gain) is **not** the primary learning-rate claim here.


## 1. Setup


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import importlib
from IPython.display import display

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "scripts").exists() else NOTEBOOK_DIR.parent
if not (PROJECT_ROOT / "scripts" / "analysis").exists():
    for parent in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
        if (parent / "scripts" / "analysis" / "rsl_rl_analysis_utils.py").exists():
            PROJECT_ROOT = parent
            break

sys.path.insert(0, str(PROJECT_ROOT / "scripts" / "analysis"))
import rsl_rl_analysis_utils as utils
importlib.reload(utils)

LOGS_DIR = PROJECT_ROOT / "logs" / "rsl_rl"
EXPORT_DIR = utils.resolve_legged_vs_wheeled_export_dir(PROJECT_ROOT / "notebooks" / "exports" / "legged_vs_wheeled")
FIG_DIR = EXPORT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = (42, 0, 1)
MAX_ITERATIONS = utils.LEGGED_VS_WHEELED_MAX_ITERATIONS
STAGE_FRACTIONS = utils.LEGGED_VS_WHEELED_STAGE_FRACTIONS
STAGE_STEPS = tuple(int(f * MAX_ITERATIONS) for f in STAGE_FRACTIONS)
EXPECTED_RUNS = 48
PRIMARY_METRICS = list(utils.LEGGED_VS_WHEELED_METRIC_COLUMNS)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

CONDITIONS = utils.LEGGED_VS_WHEELED_CONDITIONS
MORPHOLOGY_PAIRS = utils.MORPHOLOGY_PAIRS
EXPERIMENTS = list(CONDITIONS.keys())

print("PROJECT_ROOT:", PROJECT_ROOT)
print("LOGS_DIR:", LOGS_DIR)
print("EXPORT_DIR:", EXPORT_DIR)
print("Conditions:", len(EXPERIMENTS))
print("Morphology pairs:", list(MORPHOLOGY_PAIRS.keys()))
print("Stage steps:", STAGE_STEPS)


## 2. Load seed-tagged runs

Loads only folders matching `*_seed{N}` for the 16 legged-vs-wheeled experiments (seeds 42 / 0 / 1).


In [ ]:
all_metrics = utils.load_seed_tagged_metrics(
    LOGS_DIR,
    experiments=EXPERIMENTS,
    seeds=SEEDS,
)
print(f"Loaded {len(all_metrics)} seed-tagged runs")

if not all_metrics:
    print(
        "\nNo seed-tagged runs found yet.\n"
        "Train first:\n"
        "  bash scripts/experiments/run_legged_vs_wheeled_seeds.sh\n"
        "Then re-run this cell."
    )
else:
    for run_key, rd in sorted(
        all_metrics.items(), key=lambda x: (x[1]["experiment"], x[1].get("seed", -1))
    ):
        print(
            f"  seed={rd.get('seed'):>2}  {run_key}"
            f"  [{rd.get('category')}/{rd.get('pair_id')}/{rd.get('terrain')}]"
        )


## 3. Completeness check

Expect **48** runs (16 experiments × 3 seeds). Missing `(experiment, seed)` pairs are listed below.


In [ ]:
loaded_pairs = {
    (rd["experiment"], int(rd["seed"]))
    for rd in all_metrics.values()
    if rd.get("seed") is not None
}
expected_pairs = {(exp, seed) for exp in EXPERIMENTS for seed in SEEDS}
missing = sorted(expected_pairs - loaded_pairs)

print(f"Loaded: {len(all_metrics)} / {EXPECTED_RUNS} expected")
print(f"Unique (experiment, seed) pairs: {len(loaded_pairs)}")
if missing:
    print(f"\nMissing {len(missing)} run(s):")
    for exp, seed in missing:
        print(f"  {exp}  seed={seed}")
else:
    print("All 48 seed-tagged runs present.")

coverage_rows = []
for exp in EXPERIMENTS:
    seeds_found = sorted(s for e, s in loaded_pairs if e == exp)
    coverage_rows.append(
        {
            "experiment": exp,
            "display_name": CONDITIONS[exp]["display_name"],
            "pair_id": CONDITIONS[exp]["pair_id"],
            "category": CONDITIONS[exp]["category"],
            "terrain": CONDITIONS[exp]["terrain"],
            "n_seeds": len(seeds_found),
            "seeds": seeds_found,
            "complete": seeds_found == sorted(SEEDS),
        }
    )
coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)


## 4. Terminal metrics (mean ± std) + CSV / LaTeX export

Across-seed mean ± sample std (`ddof=1`) for the primary terminal metrics.


In [ ]:
per_seed_df = utils.extract_per_seed_terminal_metrics(
    all_metrics,
    metric_keys=PRIMARY_METRICS,
    include_convergence=False,  # relative steps_to_75pct is secondary; see §10
)
print("Per-seed rows:", len(per_seed_df))
if not per_seed_df.empty:
    display(per_seed_df.head(12))

group_cols = [
    c
    for c in ("experiment", "display_name", "category", "robot", "terrain", "pair_id")
    if c in per_seed_df.columns
]
agg_df = utils.aggregate_across_seeds(
    per_seed_df,
    metric_columns=PRIMARY_METRICS,
    group_cols=group_cols,
)
if not agg_df.empty:
    # Keep campaign order from LEGGED_VS_WHEELED_CONDITIONS
    order = {name: i for i, name in enumerate(EXPERIMENTS)}
    agg_df = agg_df.sort_values(
        "experiment", key=lambda s: s.map(lambda x: order.get(x, 999))
    ).reset_index(drop=True)

print("\nAggregated rows:", len(agg_df))
if not agg_df.empty:
    display(agg_df)

terminal_table = utils.build_morphology_table_dataframe(agg_df, metric_columns=PRIMARY_METRICS)
print("\n=== Terminal table (mean ± std over seeds) ===")
if terminal_table.empty:
    print("(empty — wait for training runs)")
else:
    display(terminal_table)


In [ ]:
def save_fig(fig, name: str):
    path = FIG_DIR / name
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print("Saved:", path)
    return path


if terminal_table.empty:
    print("Nothing to export yet.")
else:
    csv_path = utils.export_morphology_csv(
        terminal_table, "terminal_mean_std.csv", export_dir=EXPORT_DIR
    )
    latex = utils.export_morphology_latex(
        terminal_table,
        filename="terminal_mean_std.tex",
        export_dir=EXPORT_DIR,
        caption="Legged vs wheeled locomotion (mean $\\pm$ std over 3 seeds, 20k iterations).",
        label="tab:legged_vs_wheeled_terminal",
    )
    print("Wrote:", csv_path)
    print("Wrote:", EXPORT_DIR / "terminal_mean_std.tex")
    print("\n--- LaTeX snippet ---\n")
    print(latex)


## 5. Flat vs rough (error bars)

Same morphology on flat vs rough — mean ± seed-std bars for primary metrics.


In [ ]:
if agg_df.empty:
    print("No aggregated data yet.")
else:
    for metric, ylabel in [
        ("mean_reward", "Mean reward"),
        ("track_lin_vel", "Lin. vel. tracking"),
        ("track_ang_vel", "Ang. vel. tracking"),
        ("episode_length", "Episode length"),
    ]:
        if f"{metric}_mean" not in agg_df.columns:
            continue
        fig, ax = plt.subplots(figsize=(14, 5))
        utils.plot_seed_mean_std_bars(
            agg_df,
            metric,
            title=f"Flat vs Rough (all morphologies): {metric} (mean ± std over seeds)",
            ylabel=ylabel,
            experiments=EXPERIMENTS,
            ax=ax,
        )
        save_fig(fig, f"flat_vs_rough_{metric}.png")
        plt.show()


## 6. Paired morphology (legged vs wheeled) + wheeled − legged deltas

For each morphological pair on the same terrain, compare legged vs wheeled terminal performance and report the paired delta (wheeled − legged) as mean ± std over seeds.


In [ ]:
if agg_df.empty:
    print("No aggregated data yet.")
else:
    for pair_id, (legged_stem, wheeled_stem) in MORPHOLOGY_PAIRS.items():
        pair_exps = [
            f"{legged_stem}_flat",
            f"{legged_stem}_rough",
            f"{wheeled_stem}_flat",
            f"{wheeled_stem}_rough",
        ]
        subset = agg_df[agg_df["experiment"].isin(pair_exps)]
        if subset.empty:
            print(f"No data for pair {pair_id}")
            continue

        print(f"\n=== Pair: {pair_id} ===")
        display(utils.build_morphology_table_dataframe(subset, metric_columns=PRIMARY_METRICS))
        for metric in PRIMARY_METRICS:
            if f"{metric}_mean" not in subset.columns:
                continue
            fig, ax = plt.subplots(figsize=(9, 5))
            utils.plot_seed_mean_std_bars(
                subset,
                metric,
                title=f"{pair_id}: {metric} (legged vs wheeled, mean ± seed std)",
                experiments=pair_exps,
                ax=ax,
            )
            save_fig(fig, f"paired_{pair_id}_{metric}.png")
            plt.show()

    paired_delta_df = utils.build_morphology_paired_delta_dataframe(
        per_seed_df, metric_columns=PRIMARY_METRICS
    )
    print("\n=== Wheeled − legged deltas (mean ± std over paired seeds) ===")
    if paired_delta_df.empty:
        print("(empty)")
    else:
        show_cols = [
            "pair_id",
            "terrain",
            "legged_experiment",
            "wheeled_experiment",
            "n_seeds",
        ] + [f"{m}_mean_std" for m in PRIMARY_METRICS if f"{m}_mean_std" in paired_delta_df.columns]
        display(paired_delta_df[show_cols])

        delta_display = paired_delta_df[
            ["pair_id", "terrain"]
            + [f"{m}_mean_std" for m in PRIMARY_METRICS if f"{m}_mean_std" in paired_delta_df.columns]
        ].rename(columns={f"{m}_mean_std": m for m in PRIMARY_METRICS})
        csv_path = utils.export_morphology_csv(
            paired_delta_df, "wheeled_minus_legged_deltas.csv", export_dir=EXPORT_DIR
        )
        utils.export_morphology_latex(
            delta_display,
            filename="wheeled_minus_legged_deltas.tex",
            export_dir=EXPORT_DIR,
            caption="Wheeled minus legged terminal deltas (mean $\\pm$ std over paired seeds).",
            label="tab:legged_vs_wheeled_deltas",
        )
        print("Wrote:", csv_path)
        print("Wrote:", EXPORT_DIR / "wheeled_minus_legged_deltas.tex")


## 7. Learning-stage snapshots: $R$ and $\Delta R$ at 25 / 50 / 75 / 100% of 20k

Evaluate `mean_reward` on a common budget grid via `snapshot_at_fractions`, then stage rates via `stage_deltas_from_snapshots`. Optional AUC via `curve_auc`.


In [ ]:
# Aggregated snapshots (mean±std) for tables / markers
snap_agg = utils.snapshot_at_fractions(
    all_metrics,
    metric_key="mean_reward",
    fractions=STAGE_FRACTIONS,
    max_iterations=MAX_ITERATIONS,
)
print("Aggregated snapshots:", len(snap_agg))
if not snap_agg.empty:
    display(snap_agg.head(12))

stage_table = utils.build_morphology_stage_snapshot_dataframe(snap_agg)
print("\n=== Stage reward snapshots R_25/50/75/100 (mean ± std) ===")
if stage_table.empty:
    print("(empty)")
else:
    display(stage_table)
    utils.export_morphology_csv(stage_table, "stage_snapshots.csv", export_dir=EXPORT_DIR)
    utils.export_morphology_latex(
        stage_table,
        filename="stage_snapshots.tex",
        export_dir=EXPORT_DIR,
        caption="Mean reward at 25/50/75/100\\% of 20k iterations (mean $\\pm$ std over seeds).",
        label="tab:legged_vs_wheeled_stage_R",
    )
    print("Wrote:", EXPORT_DIR / "stage_snapshots.csv")
    print("Wrote:", EXPORT_DIR / "stage_snapshots.tex")

# Stage deltas from aggregated snapshots (Δ of means with combined std)
delta_agg = utils.stage_deltas_from_snapshots(snap_agg)
print("\nAggregated stage deltas:", len(delta_agg))
if not delta_agg.empty:
    display(delta_agg.head(12))

delta_table = utils.build_morphology_stage_delta_dataframe(delta_agg)
print("\n=== Stage learning rates ΔR (mean ± std) ===")
if delta_table.empty:
    print("(empty)")
else:
    display(delta_table)
    utils.export_morphology_csv(delta_table, "stage_deltas.csv", export_dir=EXPORT_DIR)
    utils.export_morphology_latex(
        delta_table,
        filename="stage_deltas.tex",
        export_dir=EXPORT_DIR,
        caption="Stage reward rates $\\Delta R$ over 0--25/25--50/50--75/75--100\\% of 20k (mean $\\pm$ std).",
        label="tab:legged_vs_wheeled_stage_dR",
    )
    print("Wrote:", EXPORT_DIR / "stage_deltas.csv")
    print("Wrote:", EXPORT_DIR / "stage_deltas.tex")

# Optional: also keep per-seed snapshots if downstream cells need them
snap_per_seed = utils.snapshot_at_fractions(
    all_metrics,
    metric_key="mean_reward",
    fractions=STAGE_FRACTIONS,
    max_iterations=MAX_ITERATIONS,
    per_seed=True,
)

# Normalized AUC
auc_agg = utils.curve_auc(
    all_metrics,
    metric_key="mean_reward",
    max_iterations=MAX_ITERATIONS,
    normalize=True,
)
print("\n=== Normalized AUC (mean reward) ===")
if auc_agg.empty:
    print("(empty)")
else:
    auc_table = utils.build_morphology_table_dataframe(auc_agg, metric_columns=["auc"])
    display(auc_table)
    utils.export_morphology_csv(auc_table, "curve_auc.csv", export_dir=EXPORT_DIR)
    print("Wrote:", EXPORT_DIR / "curve_auc.csv")


## 8. Stage learning-rate grouped bars

Grouped bars of stage $\Delta R$ (0–25, 25–50, 50–75, 75–100% of budget) for each condition.


In [ ]:
if delta_agg.empty:
    print("No stage-delta aggregates to plot (see §7).")
else:
    plot_df = delta_agg.copy()
    plot_df["condition"] = plot_df["display_name"]
    stages = list(dict.fromkeys(plot_df["window"].tolist()))

    # Full campaign overview
    conditions = []
    for exp in EXPERIMENTS:
        name = CONDITIONS[exp]["display_name"]
        if name in set(plot_df["condition"]) and name not in conditions:
            conditions.append(name)

    fig, ax = plt.subplots(figsize=(16, 6))
    x = np.arange(len(conditions))
    width = 0.8 / max(len(stages), 1)
    for i, stage in enumerate(stages):
        sub = plot_df[plot_df["window"] == stage].set_index("condition")
        means = [float(sub.loc[c, "mean"]) if c in sub.index else np.nan for c in conditions]
        stds = [float(sub.loc[c, "std"]) if c in sub.index and pd.notna(sub.loc[c, "std"]) else 0.0 for c in conditions]
        ax.bar(
            x + i * width - 0.4 + width / 2,
            means,
            width=width,
            yerr=stds,
            capsize=2,
            label=stage,
            alpha=0.85,
        )
    ax.set_xticks(x)
    ax.set_xticklabels(conditions, rotation=45, ha="right")
    ax.set_ylabel("ΔR (mean reward)", fontweight="bold")
    ax.set_title("Stage learning rates ΔR (mean ± seed std)", fontweight="bold")
    ax.legend(title="Stage", loc="best")
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    save_fig(fig, "stage_deltas_grouped_all.png")
    plt.show()

    # Per-pair panels
    for pair_id in MORPHOLOGY_PAIRS.keys():
        sub_pair = plot_df[plot_df["pair_id"] == pair_id]
        if sub_pair.empty:
            continue
        conds = []
        for exp in EXPERIMENTS:
            if CONDITIONS[exp]["pair_id"] != pair_id:
                continue
            name = CONDITIONS[exp]["display_name"]
            if name in set(sub_pair["condition"]) and name not in conds:
                conds.append(name)
        fig, ax = plt.subplots(figsize=(10, 5))
        x = np.arange(len(conds))
        width = 0.8 / max(len(stages), 1)
        for i, stage in enumerate(stages):
            sub = sub_pair[sub_pair["window"] == stage].set_index("condition")
            means = [float(sub.loc[c, "mean"]) if c in sub.index else np.nan for c in conds]
            stds = [
                float(sub.loc[c, "std"]) if c in sub.index and pd.notna(sub.loc[c, "std"]) else 0.0
                for c in conds
            ]
            ax.bar(
                x + i * width - 0.4 + width / 2,
                means,
                width=width,
                yerr=stds,
                capsize=3,
                label=stage,
                alpha=0.85,
            )
        ax.set_xticks(x)
        ax.set_xticklabels(conds, rotation=30, ha="right")
        ax.set_ylabel("ΔR (mean reward)", fontweight="bold")
        ax.set_title(f"{pair_id}: stage ΔR (mean ± seed std)", fontweight="bold")
        ax.legend(title="Stage", loc="best")
        ax.grid(axis="y", alpha=0.3)
        fig.tight_layout()
        save_fig(fig, f"stage_deltas_grouped_{pair_id}.png")
        plt.show()


## 9. Convergence bands with milestone markers

Mean ± seed-std training curves with vertical markers at 5k / 10k / 15k / 20k iterations (25 / 50 / 75 / 100% of budget).


In [ ]:
def _add_stage_markers(ax, steps=STAGE_STEPS):
    for step, frac in zip(steps, STAGE_FRACTIONS):
        ax.axvline(step, color="gray", linestyle="--", alpha=0.55, linewidth=1)
        ymin, ymax = ax.get_ylim()
        ax.text(
            step,
            ymax,
            f"{int(frac * 100)}%",
            rotation=90,
            va="top",
            ha="right",
            fontsize=8,
            color="gray",
        )


if not all_metrics:
    print("No runs loaded.")
else:
    fig, ax = plt.subplots(figsize=(14, 7))
    utils.plot_seed_curves_with_error_bands(
        all_metrics,
        experiments=EXPERIMENTS,
        metric_name="mean_reward",
        title="All conditions: mean reward ± seed std",
        ax=ax,
    )
    _add_stage_markers(ax)
    save_fig(fig, "convergence_all_conditions.png")
    plt.show()

    for pair_id, (legged_stem, wheeled_stem) in MORPHOLOGY_PAIRS.items():
        pair_exps = [
            f"{legged_stem}_flat",
            f"{legged_stem}_rough",
            f"{wheeled_stem}_flat",
            f"{wheeled_stem}_rough",
        ]
        fig, ax = plt.subplots(figsize=(11, 5))
        utils.plot_seed_curves_with_error_bands(
            all_metrics,
            experiments=pair_exps,
            metric_name="mean_reward",
            title=f"{pair_id}: convergence (mean ± seed std)",
            ax=ax,
        )
        _add_stage_markers(ax)
        if not snap_agg.empty:
            for exp in pair_exps:
                rows = snap_agg[snap_agg["experiment"] == exp].sort_values("fraction")
                if rows.empty:
                    continue
                ax.scatter(rows["step"], rows["mean"], s=28, zorder=5)
        save_fig(fig, f"convergence_{pair_id}.png")
        plt.show()

    for morph in ("legged", "wheeled"):
        morph_exps = [e for e in EXPERIMENTS if CONDITIONS[e]["category"] == morph]
        fig, ax = plt.subplots(figsize=(12, 6))
        utils.plot_seed_curves_with_error_bands(
            all_metrics,
            experiments=morph_exps,
            metric_name="mean_reward",
            title=f"{morph.capitalize()} morphologies: convergence (mean ± seed std)",
            ax=ax,
        )
        _add_stage_markers(ax)
        save_fig(fig, f"convergence_{morph}.png")
        plt.show()


## 10. Method notes

- **Seed-level std:** all error bars and ± values are **across seeds** (sample std, `ddof=1`), not within-run last-window confidence intervals.
- **Budget-fraction stages:** primary learning-rate analysis uses absolute milestones at 25 / 50 / 75 / 100% of $T=20000$ (steps 5k / 10k / 15k / 20k) via interpolated snapshots. Table I's relative `steps_to_75pct` (time to 75% of *that run's own* reward gain) is **not** used as the primary claim here.
- **Reward-scale caveats:** absolute `mean_reward` is not always comparable across different MDPs / robots; prefer within-pair (legged vs wheeled) and within-robot (flat vs rough) contrasts, plus tracking metrics and episode length.
- **Exports:** CSV / LaTeX / PNG under `notebooks/exports/legged_vs_wheeled/` (figures in `figures/`).
- **Train command:** `bash scripts/experiments/run_legged_vs_wheeled_seeds.sh`
